## Step 1 - Create a Realistic Synthetic Dataset

In [18]:
import numpy as np
import pandas as pd

np.random.seed(42)

n = 2000
time = pd.date_range(start='2024-01-01', periods=n, freq='H')

# Base signals (normal operations)
pressure_in = np.random.normal(100, 3, n)
flow_in = np.random.normal(500, 25, n)
temperature = np.random.normal(30, 2, n)

# Outlet values with noise
pressure_out = pressure_in - np.random.normal(5, 2, n)
flow_out = flow_in - np.random.normal(3, 3, n)

# Initialize leak label
leak = np.zeros(n)

# Introduce realistic leaks
leak_starts = np.random.choice(range(200, 1700), size=40, replace=False)

for start in leak_starts:
    duration = np.random.randint(5, 20)  # leaks persist over time
    severity = np.random.uniform(2, 8)   # smaller, realistic impact
    
    for t in range(start, min(start + duration, n)):
        leak[t] = 1
        
        # Gradual effect (not instant)
        factor = (t - start + 1) / duration
        
        pressure_out[t] -= severity * factor + np.random.normal(0, 1)
        flow_out[t] -= (severity * 2) * factor + np.random.normal(0, 2)

# Add random sensor noise (affects both classes)
pressure_out += np.random.normal(0, 1.5, n)
flow_out += np.random.normal(0, 5, n)

# Add occasional false anomalies (non-leak disturbances)
false_events = np.random.choice(range(n), size=50, replace=False)
for i in false_events:
    pressure_out[i] -= np.random.uniform(2, 5)
    flow_out[i] -= np.random.uniform(5, 10)

# Build dataframe
df = pd.DataFrame({
    'time': time,
    'pressure_in': pressure_in,
    'pressure_out': pressure_out,
    'flow_in': flow_in,
    'flow_out': flow_out,
    'temperature': temperature,
    'leak': leak
})

df.to_csv("pipeline_data.csv", index=False)

In [9]:
df.tail()

,time,pressure_in,pressure_out,flow_in,flow_out,temperature,leak
1995,2024-03-24 03:00:00,103.210451,94.070552,500.711440,494.439688,30.342939,0.0
1996,2024-03-24 04:00:00,99.920436,93.142254,448.054705,450.383252,32.305296,0.0
1997,2024-03-24 05:00:00,97.354376,92.096248,491.992555,489.555271,27.565192,0.0
1998,2024-03-24 06:00:00,99.510799,91.907596,541.084454,540.902988,30.935901,0.0
1999,2024-03-24 07:00:00,97.765292,87.334929,509.016197,496.864182,27.659439,0.0


## Step 2 - Feature Engineering

In [13]:
df['pressure_diff'] = df['pressure_in'] - df['pressure_out']
df['flow_diff'] = df['flow_in'] - df['flow_out']

# Time-based features
df['pressure_roll_mean'] = df['pressure_diff'].rolling(5).mean()
df['flow_roll_std'] = df['flow_diff'].rolling(5).std()

df = df.dropna()

df.to_csv("pipeline_data_realistic.csv", index=False)

print("Dataset created:", df.shape)

Dataset created: (1984, 11)


## Step 3 - Build the Model

In [20]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import joblib

# -----------------------------
# 1. Load Data
# -----------------------------
df = pd.read_csv("pipeline_data_realistic.csv")

# -----------------------------
# 2. Define Features & Target
# -----------------------------
features = [
    'pressure_diff',
    'flow_diff',
    'pressure_roll_mean',
    'flow_roll_std'
]

X = df[features]
y = df['leak']

# -----------------------------
# 3. Train-Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# -----------------------------
# 4. Train Model
# -----------------------------
model = RandomForestClassifier(class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

# -----------------------------
# 5. Predictions
# -----------------------------
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# -----------------------------
# 6. Evaluation (Default Threshold)
# -----------------------------
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

# -----------------------------
# 7. Confusion Matrix
# -----------------------------
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:\n", cm)

# -----------------------------
# 8. Threshold Tuning
# -----------------------------
threshold = 0.3
y_pred_custom = (y_prob > threshold).astype(int)

print("\n=== Custom Threshold (0.3) ===")
print(classification_report(y_test, y_pred_custom))

# -----------------------------
# 9. Cross Validation
# -----------------------------
cv_scores = cross_val_score(model, X, y, cv=5, scoring='roc_auc')
print("\nCross-Validated ROC-AUC:", cv_scores.mean())

# -----------------------------
# 10. Feature Importance
# -----------------------------
importance = pd.Series(model.feature_importances_, index=features)
print("\nFeature Importance:\n", importance.sort_values(ascending=False))

# -----------------------------
# 11. Save Model
# -----------------------------
joblib.dump(model, "model.pkl")
print("\nModel saved as model.pkl")

              precision    recall  f1-score   support

         0.0       0.92      0.97      0.94       318
         1.0       0.85      0.65      0.73        79

    accuracy                           0.91       397
   macro avg       0.88      0.81      0.84       397
weighted avg       0.90      0.91      0.90       397

ROC-AUC: 0.8566595016320357

Confusion Matrix:
 [[309   9]
 [ 28  51]]

=== Custom Threshold (0.3) ===
              precision    recall  f1-score   support

         0.0       0.94      0.86      0.90       318
         1.0       0.58      0.76      0.66        79

    accuracy                           0.84       397
   macro avg       0.76      0.81      0.78       397
weighted avg       0.86      0.84      0.85       397


Cross-Validated ROC-AUC: 0.8046547611064943

Feature Importance:
 pressure_roll_mean    0.338837
flow_diff             0.252615
pressure_diff         0.238150
flow_roll_std         0.170398
dtype: float64

Model saved as model.pkl
